# 11 — Silver Suppliers & Bidders

**Objetivo:** construir proveedores canónicos y una relación proceso–proponente sin forzar todavía la identidad bidder→supplier.

Este notebook conserva los identificadores necesarios para `13_entity_resolution.ipynb`, evita la expansión observada en Bronze y excluye del Silver analítico general campos de contacto/nombres sensibles que no son necesarios para la resolución determinística.

## Invariantes

- `suppliers.codigo` fue validado en M1 como PK.
- `bidders.id_procedimiento -> processes.id_del_proceso` tiene alta cobertura, pero el lado Processes requería canonicalización.
- El join directo `bidders.codigo_proveedor -> suppliers.codigo` tuvo 0% de cobertura en M1: **no se fuerza aquí**.
- `process_versions.parquet` de notebook 10 permite construir un crosswalk `procedure_id -> process_id` sin multiplicar filas.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files
from argos.config.silver import load_silver_spec
from argos.silver.suppliers_bidders import BUILD_IMPLEMENTATION

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

DATASETS, _ = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")

check10 = pd.read_parquet(paths.reports / "silver" / "10_validation.parquet")
blocking10 = check10.loc[check10["blocking"]]
assert not blocking10.empty and blocking10["passed"].all(), "Notebook 10 no está cerrado."

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)
{
    "suppliers": FILES["suppliers"].name,
    "bidders": FILES["bidders"].name,
    "implementation": BUILD_IMPLEMENTATION,
    "schema_version": SILVER_SPEC.schema_version,
    "duckdb_threads": SILVER_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": SILVER_SPEC.runtime.duckdb_memory_limit,
}

{'suppliers': 'SECOP_II_-_Proveedores_Registrados_20260921.csv',
 'bidders': 'Proponentes_por_Proceso_SECOP_II_20260921.csv',
 'implementation': 'm2-11-suppliers-bidders-v2',
 'schema_version': 'm2-silver-v1',
 'duckdb_threads': 2,
 'duckdb_memory_limit': '8GB'}

## Construcción

Salidas:

- `suppliers.parquet` — `1 fila = 1 supplier_code`.
- `bidder_participations.parquet` — preserva la población Bronze de proponentes.
- `procedure_process_map.parquet` — crosswalk sin expansión desde `procedure_id` a `process_id`.
- `bidder_process_links.parquet` — relación canónica proceso–proponente, con población ambigua/unmatched explícita.

In [2]:
from argos.silver.suppliers_bidders import build_suppliers_bidders_silver

result = build_suppliers_bidders_silver(
    FILES["suppliers"],
    FILES["bidders"],
    suppliers_spec=DATASETS["suppliers"],
    bidders_spec=DATASETS["bidders"],
    column_profile_path=paths.reports / "profiling" / "column_profile.parquet",
    reports_root=paths.reports,
    silver_dir=paths.silver,
    manifest_dir=paths.root / "artifacts" / "manifests",
    silver_spec=SILVER_SPEC,
    date_order=PROJECT_CONFIG["temporal"]["date_order"],
    sample_size=PROJECT_CONFIG["profiling"]["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)
{name: str(path.relative_to(paths.root)) for name, path in result.artifacts.items()}

{'suppliers': 'data/silver/suppliers.parquet',
 'bidder_participations': 'data/silver/bidder_participations.parquet',
 'procedure_process_map': 'data/silver/procedure_process_map.parquet',
 'bidder_process_links': 'data/silver/bidder_process_links.parquet'}

## Routing de privacidad

La exclusión es arquitectónica: los campos explícitamente sensibles no entran al Silver analítico general. Los identificadores `supplier_code`, `supplier_nit`, `bidder_code` y `bidder_nit` se conservan porque son necesarios para lineage/entity resolution, no porque sean features autorizadas.

In [3]:
result.routing.groupby(["dataset", "route"]).size().rename("columns").reset_index()

,dataset,route,columns
0,bidders,core,8
1,bidders,excluded_sensitive,1
2,suppliers,core,18
3,suppliers,excluded_sensitive,8


In [4]:
result.routing.loc[result.routing["route"] == "excluded_sensitive", [
    "dataset", "raw_column", "canonical_column", "semantic_type", "reason"
]].sort_values(["dataset", "canonical_column"])

,dataset,raw_column,canonical_column,semantic_type,reason
32,bidders,Proveedor,proveedor,category_or_text,explicit Silver privacy exclusion
11,suppliers,Correo,correo,category_or_text,explicit Silver privacy exclusion
22,suppliers,Correo representante legal,correo_representante_legal,category_or_text,explicit Silver privacy exclusion
12,suppliers,Direccion,direccion,geography,explicit Silver privacy exclusion
18,suppliers,Nombre representante legal,nombre_representante_legal,category_or_text,explicit Silver privacy exclusion
20,suppliers,Número doc representante legal,numero_doc_representante_legal,category_or_text,explicit Silver privacy exclusion
9,suppliers,Telefono,telefono,category_or_text,explicit Silver privacy exclusion
21,suppliers,Telefono representante legal,telefono_representante_legal,category_or_text,explicit Silver privacy exclusion
19,suppliers,Tipo doc representante legal,tipo_doc_representante_legal,category_or_text,explicit Silver privacy exclusion


## Validación bloqueante

In [5]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,suppliers_population_preserved,suppliers,True,True,"{""m1_expected"": 1612550, ""rows"": 1612550}",Silver suppliers preserves the Bronze supplier population
1,suppliers_primary_key,suppliers,True,True,"{""distinct_supplier_code"": 1612550, ""null_supplier_code"": 0, ""rows"": 1612550}",supplier_code is non-null and unique
2,bidder_population_preserved,bidders,True,True,"{""m1_expected"": 2316965, ""rows"": 2316965}",bidder_participations preserves every Bronze bidder row
3,bidder_process_link_grain,relationships,True,True,"{""bidder_source_rows"": 2316965, ""distinct_link_ids"": 2281440, ""links"": 2281440, ""represented_source_rows"": 2316965}",canonical bidder-process links are unique and account for all source rows
4,bidder_process_link_coverage,relationships,True,True,"{""ambiguous_links"": 0, ""ambiguous_source_rows"": 0, ""coverage"": 0.9883179072623022, ""matched_links"": 2255354, ""matched_source_rows"": 2289898, ""missing_key_source_rows"": 0, ""sour...",canonical bidder→process coverage >= 95.00%
5,party_sensitive_fields_excluded,privacy,True,True,[],configured sensitive party fields are outside general analytical Silver


In [6]:
from argos.common.duckdb import connect
from argos.common.sql import sql_string

con = connect(
    temp_directory=paths.root / ".cache" / "duckdb",
    threads=SILVER_SPEC.runtime.duckdb_threads,
    memory_limit=SILVER_SPEC.runtime.duckdb_memory_limit,
)
try:
    link_status = con.execute(f"""
        SELECT process_link_status, COUNT(*) AS links, SUM(source_row_count) AS source_rows
        FROM read_parquet({sql_string(result.artifacts['bidder_process_links'])})
        GROUP BY process_link_status ORDER BY links DESC
    """).fetchdf()
finally:
    con.close()
link_status

,process_link_status,links,source_rows
0,matched,2255354,2289898.0
1,unmatched_procedure,26086,27067.0


In [7]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()
print("11 ✅ PASS" if result.ready_for_next_notebook else "11 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))
assert result.ready_for_next_notebook, "Notebook 11 remains OPEN; resolve blocking checks before 12."

11 ✅ PASS
Blocking checks: 6/6
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/silver_suppliers_bidders.json


**Siguiente:** `12_silver_events.ipynb`. La identidad bidder→supplier sigue pendiente intencionalmente hasta `13`.